In [ ]:
# Retrieval

**RAG 流程小结**：本 notebook 是 RAG 流程第四步——**检索（Retrieval）**。前面 03 节把文档存进了向量库，
这里重点讲解多种“比朴素相似度检索更聪明”的检索策略：最大边际相关性（MMR，兼顾相关性和多样性）、
基于 metadata 的过滤检索、self-query（让 LLM 自动从自然语言问题里抽取过滤条件）、
上下文压缩检索（用 LLM 精简每个检索结果只保留和问题相关的部分），以及 SVM/TF-IDF 这类不依赖向量库的传统检索算法。

In [ ]:
import os
import openai
import sys
sys.path.append('../..')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

In [ ]:
# 【真实 Bug 修复】原代码 `from langchain.vectorstores import Chroma` 和
# `from langchain.embeddings.openai import OpenAIEmbeddings` 都是 2023 年课程的旧版路径，
# 在当前安装的 langchain 1.4.0 下会直接 ModuleNotFoundError（这两个模块都已从主包 langchain 中拆分出去）。
# 新版本正确路径分别是 langchain_community.vectorstores 和 langchain_openai（与 03 节保持一致）。
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
persist_directory = 'docs/chroma/'

In [ ]:
# 打开 03 节里已经持久化好的 Chroma 向量库：只要 persist_directory 相同、embedding 模型相同，
# 就能直接复用之前算好的向量，不用重新跑一遍 embedding
embedding = OpenAIEmbeddings()
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

In [ ]:
print(vectordb._collection.count())

In [ ]:
# 构造一个专门用来演示 MMR（最大边际相关性）效果的小例子：
# 三句话都在讲同一种蘑菇（Amanita phalloides），其中前两句语义高度重复（几乎是同一句话的不同表述）
texts = [
    """The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).""",
    """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.""",
    """A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.""",
]

In [ ]:
# from_texts：不经过 Document 对象，直接从字符串列表构造一个内存版向量库（没传 persist_directory，不落盘）
smalldb = Chroma.from_texts(texts, embedding=embedding)

In [ ]:
question = "Tell me about all-white mushrooms with large fruiting bodies"

In [ ]:
# 朴素相似度检索：因为前两句语义太接近，k=2 很可能两条结果都差不多，缺乏多样性
smalldb.similarity_search(question, k=2)

In [ ]:
# MMR（Maximal Marginal Relevance，最大边际相关性）：
# fetch_k 先取出比 k 更多的候选（这里是 3 条），再从中挑出 k=2 条——
# 挑选时不仅看和 question 的相关性，也看结果之间的差异性，尽量避免选出高度重复的内容
smalldb.max_marginal_relevance_search(question,k=2, fetch_k=3)

In [ ]:
# 回到真实的课程向量库上验证同样的问题：由于 03 节故意重复加载了 Lecture01 两次，
# 朴素相似度检索很容易在结果里返回同一段内容的两份重复
question = "what did they say about matlab?"
docs_ss = vectordb.similarity_search(question,k=3)

In [ ]:
docs_ss[0].page_content[:100]

In [ ]:
# 对比上一格：docs_ss[0] 和 docs_ss[1] 很可能几乎一模一样（重复文档导致的重复结果）
docs_ss[1].page_content[:100]

In [ ]:
# 换成 MMR 检索（这里没传 fetch_k，会用 vectorstore 的默认值），对比结果是否更多样
docs_mmr = vectordb.max_marginal_relevance_search(question,k=3)

In [ ]:
docs_mmr[0].page_content[:100]

In [ ]:
# 用 MMR 之后，docs_mmr[1] 通常会和 docs_mmr[0] 有明显差异，不再是简单重复
docs_mmr[1].page_content[:100]

### Addressing Specificity: working with metadata

In [ ]:
question = "what did they say about regression in the third lecture?"

In [ ]:
# 最直接的解决办法：手动传 filter 参数，限定 metadata 里 source 字段必须等于第三讲的文件路径，
# 这样能 100% 保证只从指定来源里检索，代价是要手写这个 filter（下面的 self-query 会让 LLM 自动生成它）
docs = vectordb.similarity_search(
    question,
    k=3,
    filter={"source":"docs/cs229_lectures/MachineLearning-Lecture03.pdf"}
)

In [ ]:
# 验证一下：所有结果的 source 应该都指向 Lecture03
for d in docs:
    print(d.metadata)

### Addressing Specificity: working with metadata using self-query retriever

In [ ]:
# 【真实 Bug 修复】self-query 检索：让 LLM 读懂问题里隐含的过滤条件（比如"第三讲"），
# 自动把问题拆成"语义查询部分" + "结构化 metadata 过滤条件"，不用像上面那样手写 filter。
#
# 原代码分别是：
#   from langchain.llms import OpenAI                                  -> ModuleNotFoundError（langchain.llms 已不存在）
#   from langchain.retrievers.self_query.base import SelfQueryRetriever -> ModuleNotFoundError（langchain.retrievers 已不存在）
#   from langchain.chains.query_constructor.base import AttributeInfo   -> ModuleNotFoundError（langchain.chains 已不存在）
# 这几个都是 2023 年课程的旧路径。在当前 langchain 1.4.0 下：
#   - 传统补全式 LLM（如 OpenAI）迁移到独立包 langchain_openai；
#   - self-query 相关的检索器和 query_constructor 被归入 langchain_classic 这个"老版本兼容层"包里。
from langchain_openai import OpenAI
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo

In [ ]:
# AttributeInfo 描述向量库里每个 metadata 字段的名字、含义、类型，
# LLM 会依据这些描述去理解"用户问题里提到的限定条件应该对应到哪个字段、怎么写过滤表达式"
metadata_field_info=[
    AttributeInfo(
        name="source",
        description="The lecture the chunk is from, should be one of `docs/cs229_lectures/MachineLearning-Lecture01.pdf`, `docs/cs229_lectures/MachineLearning-Lecture02.pdf`, or `docs/cs229_lectures/MachineLearning-Lecture03.pdf`",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="The page from the lecture",
        type="integer",
    ),
]

In [ ]:
# 【真实 Bug 修复 + 版本适配】document_content_description 描述向量库里存的是什么内容（帮助 LLM 理解语义查询部分该怎么写）。
# gpt-3.5-turbo-instruct 是补全式（非对话式）模型，用 langchain_openai.OpenAI 这个封装类来调用。
#
# 额外发现的问题：在当前 langchain_classic + langchain_community 版本组合下，
# SelfQueryRetriever.from_llm() 内部会自动探测向量库类型来选择对应的查询翻译器（structured_query_translator），
# 但这段自动探测代码引用了 `DatabricksVectorSearch`，而这个类在当前安装的 langchain_community 里已经被移除，
# 会导致 from_llm() 直接抛 ImportError（这是第三方库内部版本不兼容问题，不是本 notebook 代码的错）。
# 解决办法：绕开自动探测，显式传入 Chroma 对应的 ChromaTranslator。
from langchain_community.query_constructors.chroma import ChromaTranslator

document_content_description = "Lecture notes"
llm = OpenAI(model='gpt-3.5-turbo-instruct', temperature=0)
retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
    verbose=True,
    structured_query_translator=ChromaTranslator(),  # 见上方注释：绕开有 bug 的自动探测逻辑
)

In [ ]:
question = "what did they say about regression in the third lecture?"

In [ ]:
# 【真实 Bug 修复】get_relevant_documents 是旧版 Retriever 接口的方法，
# 在当前 langchain_core 里 BaseRetriever 已经不再提供这个公开方法（只保留内部的 _get_relevant_documents），
# 统一改用 LangChain Runnable 接口的标准方法 .invoke(...)，效果等价：传入 query 字符串，返回 List[Document]。
docs = retriever.invoke(question)

In [ ]:
# 理想情况下 SelfQueryRetriever 会自动识别出 "third lecture" -> source 过滤条件，
# 所有返回结果的 source 都应该指向 Lecture03（不需要像前面那样手写 filter）
for d in docs:
    print(d.metadata)

### Additional tricks: compression

In [ ]:
# 【真实 Bug 修复】上下文压缩检索：先正常检索出若干候选文档，再用一个 LLM 把每篇文档里和问题无关的部分删掉，
# 只保留和问题相关的片段，减少最终塞给回答模型的无关噪音、节省 token。
#
# 原代码 `from langchain_community.retrievers import ContextualCompressionRetriever` 和
# `from langchain_community.document_compressors import LLMChainExtractor` 在当前安装的
# langchain_community 0.4.2 版本下都会 ImportError（这两个类已经从 langchain_community 移到了
# langchain_classic 这个兼容层包里）。
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [ ]:
# 辅助函数：把检索到的多个 Document 按编号、分隔线的格式打印出来，方便肉眼对比压缩前后的差异
def pretty_print_docs(docs):
    print(f"\n{'-' * 100}\n".join([f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]))

In [ ]:
# LLMChainExtractor：具体执行"压缩"动作的组件，内部会调用 llm 对每篇检索到的文档做提取式摘要
# Wrap our vectorstore
llm = OpenAI(temperature=0, model="gpt-3.5-turbo-instruct")
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
# ContextualCompressionRetriever 把 base_retriever（这里是普通的向量库检索器）包了一层：
# 先用 base_retriever 正常检索，再用 base_compressor 对结果做压缩
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever()
)

In [ ]:
# 【真实 Bug 修复】同上：get_relevant_documents 已被移除，改用 .invoke(question)
question = "what did they say about matlab?"
compressed_docs = compression_retriever.invoke(question)
pretty_print_docs(compressed_docs)

## Combining various techniques

In [ ]:
# 组合技巧：把 base_retriever 换成 MMR 检索（search_type="mmr"），
# 这样既能通过 MMR 保证结果多样性，又能通过压缩去掉每条结果里的无关内容——两种技术叠加使用
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever(search_type = "mmr")
)

In [ ]:
# 【真实 Bug 修复】同上：改用 .invoke(question)
question = "what did they say about matlab?"
compressed_docs = compression_retriever.invoke(question)
pretty_print_docs(compressed_docs)

## Other types of retrieval

In [ ]:
# SVMRetriever / TFIDFRetriever：不依赖向量数据库，直接在内存里用经典机器学习算法（支持向量机 / TF-IDF）做检索，
# 适合小规模数据、快速原型验证，不需要额外部署向量库
from langchain_community.retrievers import SVMRetriever
from langchain_community.retrievers import TFIDFRetriever
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Load PDF
loader = PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture01.pdf")
pages = loader.load()
# 先把每一页拼成一整篇长文本，再统一重新切分（而不是直接对 pages 做 split_documents），
# 这样切分边界不会受原始 PDF 分页的影响
all_page_text=[p.page_content for p in pages]
joined_page_text=" ".join(all_page_text)

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1500,chunk_overlap = 150)
splits = text_splitter.split_text(joined_page_text)


In [ ]:
# SVMRetriever.from_texts 需要传入 embedding 模型（先把每段文本转成向量，再用 SVM 做检索排序）
# TFIDFRetriever.from_texts 不需要 embedding，纯粹基于词频统计（TF-IDF），完全离线、不调用任何 API
#
# 【环境限制，非代码 bug】这两个 retriever 内部都依赖 scikit-learn（sklearn），
# 但当前 venv 并未安装 scikit-learn，实际运行 from_texts() 时会抛
# `ImportError: Could not import scikit-learn, please install with pip install scikit-learn`。
# 这是缺少第三方依赖包的问题，不是 LangChain 版本升级或代码逻辑的错，如需真正跑通需要自行
# `pip install scikit-learn`（这里按要求不做静默安装，只做代码层面的静态审查）。
# Retrieve
svm_retriever = SVMRetriever.from_texts(splits,embedding)
tfidf_retriever = TFIDFRetriever.from_texts(splits)

In [ ]:
# 【真实 Bug 修复】get_relevant_documents 已被移除，改用 .invoke(question)
question = "What are major topics for this class?"
docs_svm=svm_retriever.invoke(question)
docs_svm[0]

In [ ]:
# 同样改用 .invoke(question)；可以对比 TF-IDF（关键词匹配）和前面向量检索的结果有什么不同
question = "what did they say about matlab?"
docs_tfidf=tfidf_retriever.invoke(question)
docs_tfidf[0]